In [1]:
# SimpleDirectoryReader is dynamic, detects file type and uses appropriate reader
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings, PromptTemplate
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.ingestion import IngestionPipeline
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_parse import LlamaParse

from transformers import AutoTokenizer
from transformers import pipeline as hf_pipeline
from sentence_transformers import SentenceTransformer, util

import pandas as pd, re, ast, textwrap
from datasets import Dataset

from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory
from ragas.metrics import answer_relevancy as m_answer_relevancy
from ragas.metrics import faithfulness as m_faithfulness
from ragas.metrics import context_recall as m_context_recall
from ragas.metrics import context_precision as m_context_precision
from ragas import evaluate

from transformers import pipeline as hf_pipeline

from dotenv import load_dotenv, find_dotenv

import torch
import os
import re
import csv

# --- For Azure ML Sandpit environment ---

# Project root path for Azure Sandpit environment
project_root_path = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone" 

# Change the current working directory to the project root
os.chdir(project_root_path)


# --- FIX 2: Bypass find_dotenv() and use a direct, verified path ---
dotenv_path = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/.env"

# Add a critical check to ensure the .env file exists at this path
if not os.path.exists(dotenv_path):
    raise FileNotFoundError(
        f"CRITICAL ERROR: .env file NOT FOUND at the expected path: {dotenv_path}\n"
        f"Please double-check the path you pasted into 'project_root_path'."
    )


# Load the .env file from the explicit, verified path
load_dotenv(dotenv_path=dotenv_path)

# The project root is now simply the current working directory
project_root = os.getcwd()

# --- Now, the rest of your variable loading will work correctly ---
relative_data_dir = os.getenv("VECTOR_DATASET_DIR")

# Add a check to make sure the variable was loaded successfully from the file
if not relative_data_dir:
    raise ValueError(
        "ERROR: 'VECTOR_DATASET_DIR' was not found in your .env file, or the file is empty."
    )

data_directory = os.path.join(project_root, relative_data_dir)

hf_token = os.getenv("HUGGINGFACE_TOKEN")
llama_cloud_api_key = os.getenv("LLAMA_CLOUD_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

# --- Final Verification ---
print(f"✅ Project root successfully set to: {project_root}")
print(f"✅ .env file loaded from: {dotenv_path}")
print(f"📁 Data directory set to: {data_directory}")

2025-08-24 07:31:01.770705: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-08-24 07:31:01.770800: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-08-24 07:31:02.348931: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-08-24 07:31:03.429553: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-08-24 07:31:06.655879: W tensorflow/compiler/tf2

✅ Project root successfully set to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/hj-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone
✅ .env file loaded from: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/.env
📁 Data directory set to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/hj-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/Vector_Dataset


## Vector/Graph Store Data Ingestion

In [2]:
# Check if directory exists
if not data_directory or not os.path.isdir(data_directory):
    raise ValueError(
        f"The path '{data_directory}' is not a valid directory. "
        "Please check that the VECTOR_DATASET_DIR variable is set correctly in your .env file "
        "and that the directory actually exists."
    )

# Initialize LlamaParse with your API key
llama_cloud_api_key = os.getenv("LLAMA_CLOUD_API_KEY")
if not llama_cloud_api_key:
    raise ValueError("LLAMA_CLOUD_API_KEY not found in your .env file. Please get a key from https://cloud.llamaindex.ai")

parser = LlamaParse(
    api_key=llama_cloud_api_key,
    result_type="markdown",
    verbose=True
)

# Separate the file paths based on their type (PDF vs. other)
pdf_filepaths = []
other_filepaths = []
for filename in os.listdir(data_directory):
    file_path = os.path.join(data_directory, filename)
    if os.path.isfile(file_path):
        if filename.lower().endswith('.pdf'):
            pdf_filepaths.append(file_path)
        else:
            other_filepaths.append(file_path)

print(f"--- Found {len(pdf_filepaths)} PDF(s) and {len(other_filepaths)} other file(s) to process. ---")

# Process the files in batches
all_documents = []

# Process all PDFs in a single batch call to LlamaParse
if pdf_filepaths:
    print("\n- Parsing PDF files with LlamaParse...")
    try:
        # Calling parser.load_data() with a LIST of files is the correct way
        pdf_docs = parser.load_data(pdf_filepaths)
        all_documents.extend(pdf_docs)
        print(f"  -> Successfully parsed {len(pdf_filepaths)} PDF file(s).")
    except Exception as e:
        print(f"  -> FAILED to parse PDFs with LlamaParse. Error: {e}")

# Process all other files in a single batch call to SimpleDirectoryReader
if other_filepaths:
    print("\n- Parsing other files with SimpleDirectoryReader...")
    try:
        other_docs = SimpleDirectoryReader(input_files=other_filepaths).load_data()
        all_documents.extend(other_docs)
        print(f"  -> Successfully parsed {len(other_filepaths)} other file(s).")
    except Exception as e:
        print(f"  -> FAILED to parse other files. Error: {e}")

# The 'documents' variable should now contain all chunks from all parsed files
documents = all_documents
print(f"\n--- Ingestion complete ---")
print(f"Successfully loaded and chunked a total of {len(documents)} document(s) from all files in '{data_directory}'.")


--- Found 8 PDF(s) and 0 other file(s) to process. ---

- Parsing PDF files with LlamaParse...


Parsing files:   0%|          | 0/8 [00:00<?, ?it/s]

Started parsing the file under job_id 62857752-0f14-4100-9278-ea8099264f9a
Started parsing the file under job_id a5642226-6afc-4cef-92b2-2a30c3ddc720
Started parsing the file under job_id 281adf35-6aac-4fcc-b9bd-1262a6520db8
Started parsing the file under job_id 0aa3e01f-55b6-4142-b3ac-64acfecd8064


Parsing files:  25%|██▌       | 2/8 [00:07<00:19,  3.31s/it]

Started parsing the file under job_id c79c9c35-aaad-4e68-ae2e-8caf3a92d171
Started parsing the file under job_id d64f1573-b441-4449-8bca-97f3914f18d9


Parsing files:  50%|█████     | 4/8 [00:13<00:11,  2.86s/it]

Started parsing the file under job_id c9a00556-9840-40f0-9ee5-ae6e197bc43a
Started parsing the file under job_id a29093ed-b4d3-419d-857f-e169ba1bc45f


Parsing files: 100%|██████████| 8/8 [00:27<00:00,  3.42s/it]

  -> Successfully parsed 8 PDF file(s).

--- Ingestion complete ---
Successfully loaded and chunked a total of 137 document(s) from all files in '/mnt/batch/tasks/shared/LS_root/mounts/clusters/hj-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/Vector_Dataset'.


## Llama 3.1 8B Instruct

In [3]:
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)

# Initialize the LLM with corrected parameters to prevent errors and allow for natural answers.
llm = HuggingFaceLLM(
    model_name=model_name,
    tokenizer_name=model_name,
    device_map="auto",
    max_new_tokens=1024,  # Set max_new_tokens here to avoid conflicts
    model_kwargs={"token": hf_token, "torch_dtype": torch.bfloat16},
    generate_kwargs={
        "temperature": 0.1,
        "do_sample": True,
        # We REMOVE eos_token_id from here to allow for natural, complete sentences.
    }
)

print("HuggingFaceLLM initialized for conversational responses.")


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

HuggingFaceLLM initialized for conversational responses.


## Vector Embeddings

In [4]:
Settings.llm = llm
Settings.embed_model = "local:BAAI/bge-small-en-v1.5"

print("Global settings configured with Llama 3.1 and bge-small embedding model.")

Global settings configured with Llama 3.1 and bge-small embedding model.


In [5]:
# Pass the list 'documents' directly, creating separate index entries for each document chunk
# By default, VectorStoreIndex chunk size is set to 1024 characters
# And chunk overlap is set to 20% of the chunk size
# Each chunk will be 1024 characters long with a 205 character overlap
node_parser = SentenceSplitter(
    chunk_size=1024,
    chunk_overlap=200
)

pipeline = IngestionPipeline(
    transformations=[node_parser]
)

# Run your documents through the pipeline to create the custom-chunked nodes.
nodes = pipeline.run(documents=documents)

index = VectorStoreIndex(nodes)

print(f"Vector store index has been built successfully from {len(documents)} source document(s).")
print(f"Total nodes created with custom chunking: {len(nodes)}")
print(f"Chunk size: {node_parser.chunk_size}, Chunk overlap: {node_parser.chunk_overlap}")


Vector store index has been built successfully from 137 source document(s).
Total nodes created with custom chunking: 137
Chunk size: 1024, Chunk overlap: 200


## Retrieve Answer from Datastore

In [6]:
query_text_vector = "What is the total number of physical crime cases in 2023?"

# Retriever to get the top 3 most similar nodes from the index
retriever = index.as_retriever(similarity_top_k=3)
retrieved_nodes = retriever.retrieve(query_text_vector)

raw_chunks = [n.get_content() for n in retrieved_nodes]

seen = set()
cleaned_chunks = []

for txt in raw_chunks:
    # Strip known cues that cause echoing
    t = txt.replace("(1 sentence)", "").strip()

    # De-duplicate identical chunks
    if t and t not in seen:
        cleaned_chunks.append(t)
        seen.add(t)

if not cleaned_chunks:
    cleaned_chunks = [txt.replace("(1 sentence)", "").strip() for txt in raw_chunks if txt.strip()]

# Combine the content of the retrieved nodes into a single context string
# Contains the "exact answer" material for the LLM
exact_context = "\n\n---\n\n".join(cleaned_chunks)

conversational_prompt_template = PromptTemplate(
    "You are a precise Q&A assistant. Use ONLY the context to answer the single question. "
    "Do not invent or add anything not present in the context. Do not repeat the question. "
    "Do not answer any other questions. Output exactly one sentence and then stop.\n\n"
    "Context:\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n\n"
    "User's Question: {query_str}\n"
    "Answer: "
)

# Format the prompt with our retrieved context and the original query
final_prompt = conversational_prompt_template.format(
    context_str=exact_context,
    query_str=query_text_vector
)

# Generate with a tight token cap
raw_response = llm.complete(
    final_prompt,
    max_new_tokens=64
)

# Trim to first sentence to prevent duplication or rambling
def first_sentence(text: str) -> str:
    s = text.strip()
    # Simple split on period
    parts = s.split(".")
    return (parts[0] + ".").strip() if parts and parts else s

extracted_answer = first_sentence(str(raw_response))
print(extracted_answer)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


19,966.


## Transform Extracted Answer To Be Conversational

In [7]:
# Normalize the extracted answer
# Eg "19,966." -> "19,966"
answer_core = extracted_answer.strip()
answer_core = re.sub(r"\.\s*$", "", answer_core).strip()

# Failsafe for empty string
if not answer_core:
    print("Sorry, I couldn’t extract an answer from the context.")
else:
    # Tightly constrained one-sentence rephrase prompt that MUST include the extracted answer once
    REPHRASE_PROMPT = (
        "You are a precise assistant. Write exactly ONE conversational sentence that answers the question.\n"
        "Hard constraints:\n"
        "- Use the Extracted answer exactly once.\n"
        "- Do not add any other information not present in the Extracted answer or the Question.\n"
        "- Do not repeat yourself.\n"
        "- End with a single period.\n\n"
        f"Extracted answer: {answer_core}\n"
        f"Question: {query_text_vector}\n"
        "Answer:"
    )

    # Generate a short rephrased sentence to avoid loops
    rephrase_raw = llm.complete(REPHRASE_PROMPT, max_new_tokens=32)

    # Collapse whitespace
    s = " ".join(str(rephrase_raw).strip().split())

    # Remove a potential leading echo of the bare answer (e.g., "19,966. The total ...")
    if answer_core:
        s = re.sub(rf"^\s*{re.escape(answer_core)}\.\s*", "", s).strip()

    # Ensure exactly one sentence
    idx = s.find(".")
    s = (s[: idx + 1] if idx != -1 else s + ".").strip()

    # Enforce inclusion of the extracted answer exactly once
    # If the model dropped the value, fall back to the minimal guaranteed version
    if answer_core not in s:
        s = f"{answer_core}."

    print(s)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


The total number of physical crime cases in 2023 is 19,966.


## Benchmarking

In [17]:
# --- Config paths ---
relative_benchmark_path = os.getenv("VECTOR_BENCHMARK_DATASET_DIR")
if not relative_benchmark_path:
    raise ValueError("VECTOR_BENCHMARK_DATASET_DIR not set in .env")
BENCHMARK_FILE_PATH = os.path.join(project_root, relative_benchmark_path)
OUTPUT_FILENAME = "(test)vector_benchmark_results.csv"
OUTPUT_FILE_PATH = os.path.join(os.getcwd(), OUTPUT_FILENAME)

# --- Helper Functions ---
def clean_list_str(data):
    """Ensure all items in a list are non-empty, stripped strings."""
    return [s.strip() for s in data if isinstance(s, str) and s.strip()]

def one_sentence_guard(text: str) -> str:
    """Collapse whitespace and truncate to the first sentence."""
    s = " ".join(str(text).strip().split())
    if "." in s:
        s = s.split(".")[0].strip() + "."
    # Limit max length to prevent run-on sentences
    return (s[:250] + "." if len(s) > 250 else s) if s else ""

def create_short_prompt(context_chunks, question, max_chars=400, max_chunks=3):
    """Create a concise prompt to accelerate LLM generation."""
    trimmed_chunks = []
    for chunk in context_chunks[:max_chunks]:
        trimmed_chunks.append(chunk[:max_chars] + "..." if len(chunk) > max_chars else chunk)
    
    context_str = "\n\n---\n\n".join(trimmed_chunks)
    
    return (
        "Use ONLY the provided context to answer the question. Output ONE single, concise sentence.\n\n"
        f"Context:\n---------------------\n{context_str}\n---------------------\n\n"
        f"Question: {question}\nAnswer:"
    )

# --- Load and Prepare Data ---
if not os.path.isfile(BENCHMARK_FILE_PATH):
    raise FileNotFoundError(f"Benchmark file not found at: {BENCHMARK_FILE_PATH}")

rows = []
with open(BENCHMARK_FILE_PATH, "r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    if not {"Question", "Answer"}.issubset(reader.fieldnames):
        raise ValueError("CSV must have 'Question' and 'Answer' columns.")
    rows.extend(reader)
print(f"Loaded {len(rows)} question-answer pairs for evaluation.")

# Load the benchmark CSV as a DataFrame (retains original columns: Question, Answer, Context)
benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH)

# Slice to first 3 rows ONLY (for testing; remove or adjust for full run)
benchmark_df = benchmark_df.head(3)

# Add new columns to the DataFrame (initialize as empty)
benchmark_df['retrieved_contexts'] = None
benchmark_df['response'] = None

# --- Generate Predictions ---
ragas_data = {"question": [], "answer": [], "contexts": [], "ground_truth": []}
retriever = index.as_retriever(similarity_top_k=3)
valid_samples, skipped_samples = 0, 0

print("Generating predictions and context for each question...")
for i, row in benchmark_df.iterrows():
    question = row['Question'].strip()
    ground_truth = row['Answer'].strip()
    
    # Retrieve contexts
    nodes = retriever.retrieve(question)
    contexts = [n.get_content().strip() for n in nodes if n.get_content().strip()]  # Clean list
    
    # Generate response
    prompt = create_short_prompt(contexts, question)
    raw_response = llm.complete(prompt, max_new_tokens=48)
    prediction = one_sentence_guard(str(raw_response))
    
    # Store in DataFrame: Join contexts for clean, readable string in CSV
    benchmark_df.at[i, 'retrieved_contexts'] = "\n\n---\n\n".join(contexts)
    benchmark_df.at[i, 'response'] = prediction
    
    # Collect for Ragas (use raw list for evaluation)
    ragas_data["question"].append(question)
    ragas_data["answer"].append(prediction)
    ragas_data["contexts"].append(contexts)  # List for Ragas metrics
    ragas_data["ground_truth"].append(ground_truth)
    valid_samples += 1

    if i % 10 == 0 or i == len(rows):
        print(f"Processed {i}/{len(rows)} -> Valid: {valid_samples}, Skipped: {skipped_samples}")

if valid_samples == 0:
    raise RuntimeError("No valid samples were generated. Check your data and retriever.")

ragas_dataset = Dataset.from_dict(ragas_data)
print(f"\nPrepared {len(ragas_dataset)} valid samples for Ragas evaluation.")

judge_llm = llm_factory(model="gpt-4o") 
judge_embeddings = embedding_factory(model="text-embedding-ada-002")

# Faithfulness -> how grounded a generated answer is to the provided source of information
# between 0 and 1 with 1 being fully supported by retrieved context and 0 being not supported at all

# Correctness -> factual similarity and semantic alignment
# answer generated by llm contain same key information and facts even if wording is different

# Accuracy -> exact match
answer_relevancy = m_answer_relevancy.__class__(llm=judge_llm, embeddings=judge_embeddings)
faithfulness = m_faithfulness.__class__(llm=judge_llm)
context_recall = m_context_recall.__class__(llm=judge_llm)
context_precision = m_context_precision.__class__(llm=judge_llm)

metrics_to_evaluate = [answer_relevancy, faithfulness, context_recall, context_precision]

print("\nRunning Ragas evaluation with Ragas-native models bound into metrics...")
result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics_to_evaluate,
    llm=judge_llm,
    embeddings=judge_embeddings,
)

print("Ragas evaluation complete.")

# --- Format and Save Results ---
scores_df = result.to_pandas()

# Drop initialized metric columns from benchmark_df to avoid overlap during join
cols_to_drop = ['faithfulness', 'answer_relevancy', 'context_recall', 'context_precision']
for col in cols_to_drop:
    if col in benchmark_df.columns:
        benchmark_df = benchmark_df.drop(columns=[col])

# Reset indexes and join safely
benchmark_df = benchmark_df.reset_index(drop=True)
scores_df = scores_df.reset_index(drop=True)
benchmark_df = benchmark_df.join(scores_df[cols_to_drop])

# Apply cleaning to 'retrieved_contexts' (now it will be a single sentence)
benchmark_df['retrieved_contexts'] = benchmark_df['retrieved_contexts'].apply(clean_context)

# Desired order: Original 3 + retrieved_contexts + response + metrics
desired_columns = ['Question', 'Answer', 'Context', 'retrieved_contexts', 'response', 'faithfulness', 'answer_relevancy', 'context_recall', 'context_precision']
benchmark_df = benchmark_df[desired_columns]

# Save the formatted CSV
benchmark_df.to_csv(OUTPUT_FILE_PATH, index=False)
print(f"Detailed results saved to: {OUTPUT_FILE_PATH}")

print("\n--- Overall Ragas Performance Metrics (Averages) ---")
print(scores_df.mean(numeric_only=True))
print("----------------------------------------------------")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Loaded 100 question-answer pairs for evaluation.
Generating predictions and context for each question...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed 0/100 -> Valid: 1, Skipped: 0


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



Prepared 3 valid samples for Ragas evaluation.

Running Ragas evaluation with Ragas-native models bound into metrics...


Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

Ragas evaluation complete.
Detailed results saved to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/hj-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/(test)vector_benchmark_results.csv

--- Overall Ragas Performance Metrics (Averages) ---
answer_relevancy     0.534177
faithfulness         0.666667
context_recall       1.000000
context_precision    1.000000
dtype: float64
----------------------------------------------------
